<a href="https://colab.research.google.com/github/ajsarsva/video-captioning-thesis/blob/main/notebooks/23_T5Fusion_ABC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('/content/drive/MyDrive/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

if os.path.exists('/content/video-captioning-thesis'):
    %cd /content/video-captioning-thesis
    !git pull origin main
else:
    !git clone https://github.com/ajsarsva/video-captioning-thesis.git
    %cd /content/video-captioning-thesis

import sys
sys.path.append('/content/video-captioning-thesis/src')

print("✅ Ready!")

Mounted at /content/drive
Cloning into 'video-captioning-thesis'...
remote: Enumerating objects: 113, done.
remote: Counting objects: 100% (113/113), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 113 (delta 61), reused 27 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (113/113), 19.92 MiB | 15.50 MiB/s, done.
Resolving deltas: 100% (61/61), done.
/content/video-captioning-thesis
✅ Ready!


Setup

In [2]:
!pip install transformers accelerate sentencepiece -q

import os, json, time, sys
import numpy as np
import torch
import cv2
from PIL import Image

sys.path.append('/content/video-captioning-thesis/src')

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print("✅ Setup complete!")

GPU: Tesla T4
✅ Setup complete!


 Load All Saved Data

In [3]:
# Load all three strategy results (for keyframe indices)
with open('/content/drive/MyDrive/thesis-data/captions/strategy_A_full.json') as f:
    data_a = json.load(f)
with open('/content/drive/MyDrive/thesis-data/captions/strategy_B_full.json') as f:
    data_b = json.load(f)
with open('/content/drive/MyDrive/thesis-data/captions/strategy_C_full.json') as f:
    data_c = json.load(f)

# Load ground truth
with open('/content/drive/MyDrive/thesis-data/MSR_VTT.json') as f:
    gt_data = json.load(f)

ground_truth = {}
for ann in gt_data['annotations']:
    vid = ann['image_id']
    if vid not in ground_truth:
        ground_truth[vid] = []
    ground_truth[vid].append(ann['caption'])

# Select 250 videos that have valid indices in all three strategies
sample_video_ids = []
for i in range(7010):
    vid = f'video{i}'
    if (vid in data_a and
        vid in data_b and
        vid in data_c and
        len(data_a[vid].get('keyframe_indices', [])) > 0 and
        len(data_b[vid].get('keyframe_indices', [])) > 0 and
        len(data_c[vid].get('keyframe_indices', [])) > 0):
        sample_video_ids.append(vid)
    if len(sample_video_ids) == 250:
        break

print(f"Selected {len(sample_video_ids)} videos")
print(f"Sample: {sample_video_ids[:5]}")

Selected 250 videos
Sample: ['video0', 'video1', 'video2', 'video3', 'video4']


Load Models

In [4]:
from transformers import (BlipProcessor, BlipForConditionalGeneration,
                          AutoTokenizer, AutoModelForSeq2SeqLM)

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load BLIP
print("Loading BLIP...")
blip_processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base").to(device)
print("✅ BLIP ready!")

# Load FLAN-T5
print("Loading FLAN-T5...")
t5_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
t5_model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-base").to(device)
print("✅ FLAN-T5 ready!")

print(f"\nAll models on {device}!")

Loading BLIP...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ FLAN-T5 ready!

All models on cuda!


 Helper Functions

In [5]:
from frame_extractor import extract_frames

def caption_single_frame(frame):
    """Generate BLIP caption for one frame."""
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    image = Image.fromarray(frame_rgb)
    inputs = blip_processor(image, return_tensors="pt").to(device)
    with torch.no_grad():
        output = blip_model.generate(**inputs, max_new_tokens=50)
    return blip_processor.decode(output[0], skip_special_tokens=True)


def t5_fusion(captions):
    """Fuse multiple captions into one coherent description."""
    if len(captions) == 0:
        return ""
    if len(captions) == 1:
        return captions[0]

    # Remove duplicates while preserving order
    unique_captions = list(dict.fromkeys(captions))

    prompt = (
        f"These are descriptions of different frames from the same video: "
        f"{'. '.join(unique_captions)}. "
        f"Write one concise sentence describing what is happening in the video:"
    )

    inputs = t5_tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = t5_model.generate(
            **inputs,
            max_new_tokens=50,
            num_beams=4,
            early_stopping=True
        )

    return t5_tokenizer.decode(outputs[0], skip_special_tokens=True)


def run_fusion_strategy(video_ids, keyframe_data, video_dir,
                        save_path, strategy_name):
    """
    Run T5 fusion captioning for a strategy using saved keyframe indices.

    Args:
        video_ids: list of video IDs to process
        keyframe_data: loaded JSON with keyframe_indices per video
        video_dir: path to video files
        save_path: where to save results on Drive
        strategy_name: name for progress printing
    """
    # Load checkpoint if exists
    if os.path.exists(save_path):
        with open(save_path) as f:
            results = json.load(f)
        print(f"Resuming {strategy_name}: {len(results)} done")
    else:
        results = {}
        print(f"Starting {strategy_name} fresh")

    remaining = [v for v in video_ids if v not in results]
    print(f"Remaining: {len(remaining)}")

    errors = []
    start_time = time.time()

    for i, video_name in enumerate(remaining):
        video_path = os.path.join(video_dir, f'{video_name}.mp4')

        try:
            # Load saved keyframe indices
            indices = keyframe_data[video_name]['keyframe_indices']

            # Extract video frames
            frames, fps, total = extract_frames(video_path)

            # Get valid frames from saved indices
            valid_frames = [frames[idx] for idx in indices
                           if idx < len(frames)]

            if len(valid_frames) == 0:
                raise ValueError("No valid frames found")

            # Caption each frame individually with BLIP
            frame_captions = [caption_single_frame(f) for f in valid_frames]

            # Fuse all captions with T5
            final_caption = t5_fusion(frame_captions)

            results[video_name] = {
                'caption': final_caption,
                'frame_captions': frame_captions,
                'keyframe_indices': indices,
                'num_frames_captioned': len(valid_frames),
                'total_frames': total
            }

        except Exception as e:
            errors.append(video_name)
            results[video_name] = {
                'caption': '',
                'frame_captions': [],
                'keyframe_indices': [],
                'error': str(e)
            }

        # Save checkpoint every 50 videos
        if (i+1) % 50 == 0:
            with open(save_path, 'w') as f:
                json.dump(results, f, indent=2)
            elapsed = time.time() - start_time
            rate = (i+1) / elapsed
            eta = (len(remaining) - (i+1)) / rate / 60
            print(f"  {len(results)}/250 | "
                  f"{elapsed/60:.1f} min | "
                  f"ETA: {eta:.1f} min")

    # Final save
    with open(save_path, 'w') as f:
        json.dump(results, f, indent=2)

    elapsed = time.time() - start_time
    good = len([v for v in results if results[v].get('caption', '').strip()])
    print(f"\n{strategy_name} complete!")
    print(f"Total time: {elapsed/60:.1f} min")
    print(f"Avg per video: {elapsed/250:.2f} sec")
    print(f"Good captions: {good}/1000")
    print(f"Errors: {len(errors)}")

    return results

Run Strategy A+ (Uniform + T5 Fusion)

In [6]:
video_dir = '/content/drive/MyDrive/thesis-data/videos/'

results_aplus = run_fusion_strategy(
    video_ids=sample_video_ids,
    keyframe_data=data_a,
    video_dir=video_dir,
    save_path='/content/drive/MyDrive/thesis-data/captions/strategy_Aplus_1000.json',
    strategy_name='Strategy A+'
)

Starting Strategy A+ fresh
Remaining: 250
  50/250 | 4.0 min | ETA: 15.9 min
  100/250 | 7.4 min | ETA: 11.1 min
  150/250 | 10.9 min | ETA: 7.3 min
  200/250 | 14.4 min | ETA: 3.6 min
  250/250 | 18.0 min | ETA: 0.0 min

Strategy A+ complete!
Total time: 18.0 min
Avg per video: 4.31 sec
Good captions: 250/1000
Errors: 0


Run Strategy B+ (SSIM + T5 Fusion)

In [7]:
results_bplus = run_fusion_strategy(
    video_ids=sample_video_ids,
    keyframe_data=data_b,
    video_dir=video_dir,
    save_path='/content/drive/MyDrive/thesis-data/captions/strategy_Bplus_1000.json',
    strategy_name='Strategy B+'
)

Starting Strategy B+ fresh
Remaining: 250
  50/250 | 1.9 min | ETA: 7.6 min
  100/250 | 3.8 min | ETA: 5.7 min
  150/250 | 5.6 min | ETA: 3.7 min
  200/250 | 7.8 min | ETA: 1.9 min
  250/250 | 9.7 min | ETA: 0.0 min

Strategy B+ complete!
Total time: 9.7 min
Avg per video: 2.33 sec
Good captions: 250/1000
Errors: 0


 Run Strategy C+ (CLIP + T5 Fusion)


In [8]:
results_cplus = run_fusion_strategy(
    video_ids=sample_video_ids,
    keyframe_data=data_c,
    video_dir=video_dir,
    save_path='/content/drive/MyDrive/thesis-data/captions/strategy_Cplus_1000.json',
    strategy_name='Strategy C+'
)

Starting Strategy C+ fresh
Remaining: 250
  50/250 | 2.3 min | ETA: 9.3 min
  100/250 | 4.6 min | ETA: 6.9 min
  150/250 | 6.8 min | ETA: 4.6 min
  200/250 | 9.3 min | ETA: 2.3 min
  250/250 | 11.8 min | ETA: 0.0 min

Strategy C+ complete!
Total time: 11.8 min
Avg per video: 2.82 sec
Good captions: 250/1000
Errors: 0


Quick Sanity Check

In [9]:
# Load existing single-frame results for same 1000 videos
with open('/content/drive/MyDrive/thesis-data/captions/strategy_A_full.json') as f:
    results_a_full = json.load(f)
with open('/content/drive/MyDrive/thesis-data/captions/strategy_B_full.json') as f:
    results_b_full = json.load(f)
with open('/content/drive/MyDrive/thesis-data/captions/strategy_C_full.json') as f:
    results_c_full = json.load(f)

print("CAPTION COMPARISON — Single frame vs T5 Fusion")
print("=" * 70)

for vid in sample_video_ids[:5]:
    print(f"\n{vid}:")
    print(f"  A  (single): {results_a_full[vid]['caption']}")
    print(f"  A+ (fusion): {results_aplus[vid]['caption']}")
    print(f"  B  (single): {results_b_full[vid]['caption']}")
    print(f"  B+ (fusion): {results_bplus[vid]['caption']}")
    print(f"  C  (single): {results_c_full[vid]['caption']}")
    print(f"  C+ (fusion): {results_cplus[vid]['caption']}")
    refs = ground_truth[vid][:2]
    print(f"  REF1: {refs[0]}")
    print(f"  REF2: {refs[1]}")

CAPTION COMPARISON — Single frame vs T5 Fusion

video0:
  A  (single): a road with a sign on it
  A+ (fusion): A man drives a car down a road at night.
  B  (single): a car driving down a road at night
  B+ (fusion): There is a person riding a bike down a road.
  C  (single): audi q q spotted in the wild
  C+ (fusion): A man is driving a car with a woman in the passenger seat.
  REF1: a car is shown
  REF2: a group is dancing

video1:
  A  (single): a person is cooking in a pot
  A+ (fusion): Two women are preparing food in a kitchen.
  B  (single): a person is stirring food into a pot
  B+ (fusion): A person is putting food into a pot. A person is putting food into a pan. A person is stirring food into a pot.
  C  (single): a video of a pot of soup being cooked
  C+ (fusion): A woman is washing her hands in the bathroom. A woman is making food in a kitchen.
  REF1: in a kitchen a woman adds different ingredients into the pot and stirs it
  REF2: a woman puts prawns and seasonings into

Save Video IDs Used

In [10]:
# Save the 1000 video IDs so evaluation notebook uses exact same set
with open('/content/drive/MyDrive/thesis-data/results/fusion_sample_ids.json', 'w') as f:
    json.dump(sample_video_ids, f)

print("Sample video IDs saved!")
print(f"A+ captions: {len(results_aplus)}")
print(f"B+ captions: {len(results_bplus)}")
print(f"C+ captions: {len(results_cplus)}")
print("\nAll caption files saved to Drive!")
print("Ready for LLM evaluation notebook!")

Sample video IDs saved!
A+ captions: 250
B+ captions: 250
C+ captions: 250

All caption files saved to Drive!
Ready for LLM evaluation notebook!
